# ⚡ XGBOOST CLASSIFIER (GRADIENT BOOSTING)

> **🎯 AMAÇ:** Ardışık ağaçlarla hataları düzeltererek yüksek doğruluk elde etmek  
> **📥 GİRDİ:** X_train, X_test, y_train, y_test (bellekten)  
> **📤 ÇIKTI:** y_pred, Accuracy, Confusion Matrix, Classification Report, Feature Importance  

### ⏱️ NE ZAMAN KULLANILIR?
- Tablo (structured/tabular) verisi + sınıflandırma görevlerinde
- Kaggle / yarışmalarda maksimum skor için
- Hem sınıflandırma hem regresyon için (XGBRegressor)
- Eksik değerlerle otomatik başa çıkabilir

### 🔧 TEMEL PARAMETRELER
| Parametre | Varsayılan | Açıklama |
|-----------|-----------|----------|
| `n_estimators` | 100 | Ağaç sayısı — arttırdıkça güçlenir, yavaşlar |
| `learning_rate` | 0.3 | Adım büyüklüğü — küçük = daha sağlam ama yavaş |
| `max_depth` | 6 | Ağaç derinliği — büyük = overfit riski |
| `subsample` | 1.0 | Her ağaç için kullanılacak veri oranı |
| `colsample_bytree` | 1.0 | Her ağaç için kullanılacak özellik oranı |
| `min_child_weight` | 1 | Yaprak düğümünde minimum örnek sayısı |
| `gamma` | 0 | Bölünme için min. kazanç eşiği |
| `reg_alpha` | 0 | L1 regularization |
| `reg_lambda` | 1 | L2 regularization |
| `eval_metric` | 'logloss' | Değerlendirme metriği |

---
### ⚠️ UYARI
X_train, X_test, y_train, y_test bellekte olmali.  
Kurulum: `pip install xgboost`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold
print('=' * 55)
print('⚡ XGBOOST CLASSIFIER BAŞLATILIYOR')
print('=' * 55)
print(f'XGBoost version: {xgb.__version__}')
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

In [ ]:
# 1. MODEL TANIMI
xgb_clf = xgb.XGBClassifier(
    n_estimators      = 200,    # agac sayisi
    learning_rate     = 0.1,    # kucuk = saglamama ama yavas
    max_depth         = 5,      # derinlik
    subsample         = 0.8,    # her agac %80 veri
    colsample_bytree  = 0.8,    # her agac %80 feature
    min_child_weight  = 3,      # yaprak min. ornek
    gamma             = 0.1,    # bolunme icin min. kazanc
    reg_alpha         = 0.0,    # L1
    reg_lambda        = 1.0,    # L2
    eval_metric       = 'logloss',
    random_state      = 42,
    n_jobs            = -1
)

# 2. EGİT
y_train_fit = y_train.values.ravel() if hasattr(y_train, 'values') else y_train
y_test_fit  = y_test.values.ravel()  if hasattr(y_test,  'values') else y_test
xgb_clf.fit(X_train, y_train_fit, eval_set=[(X_test, y_test_fit)], verbose=50)
print('\n✅ XGBoost modeli egitildi.')

In [ ]:
# 3. TAHMİN & METRİKLER
y_pred = xgb_clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f'\n🎯 Accuracy: %{acc*100:.2f}')
print('\n📋 Classification Report:')
print('-' * 55)
print(classification_report(y_test, y_pred))

In [ ]:
# 4. CONFUSION MATRIX GORSELLEsTİRME
cm      = confusion_matrix(y_test, y_pred)
cm_norm = confusion_matrix(y_test, y_pred, normalize='true')
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', linewidths=0.5,
            annot_kws={'size': 14, 'weight': 'bold'}, ax=axes[0])
axes[0].set_title(f'Confusion Matrix (Ham)\nAccuracy: %{acc*100:.2f}', fontweight='bold')
axes[0].set_xlabel('Tahmin'); axes[0].set_ylabel('Gercek')
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='YlOrRd',
            linewidths=0.5, vmin=0, vmax=1, ax=axes[1])
axes[1].set_title('Confusion Matrix (Normalize)', fontweight='bold')
axes[1].set_xlabel('Tahmin'); axes[1].set_ylabel('Gercek')
plt.suptitle('⚡ XGBoost - Confusion Matrix Analizi', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# 5. FEATURE IMPORTANCE (3 YONTEM)
feature_names = list(X_train.columns) if hasattr(X_train,'columns') else [f'f{i}' for i in range(X_train.shape[1])]
importance_types = {'weight':'Kullantim Sayisi','gain':'Ortalama Kazanc','cover':'Kapsama'}
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, (imp_type, title) in zip(axes, importance_types.items()):
    scores = xgb_clf.get_booster().get_score(importance_type=imp_type)
    df_imp = pd.DataFrame({'feature': list(scores.keys()), 'score': list(scores.values())})
    df_imp = df_imp.sort_values('score', ascending=False).head(15)
    colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(df_imp)))[::-1]
    ax.barh(df_imp['feature'][::-1], df_imp['score'][::-1], color=colors, edgecolor='white')
    ax.set_title(title, fontweight='bold')
plt.suptitle('⚡ XGBoost - Feature Importance (3 Yontem)', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# 6. CROSS-VALIDATION
import warnings; warnings.filterwarnings('ignore')
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_model = xgb.XGBClassifier(n_estimators=200, learning_rate=0.1, max_depth=5,
    subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42, n_jobs=-1)
cv_scores = cross_val_score(cv_model, X_train, y_train_fit, cv=skf, scoring='accuracy', n_jobs=-1)
print('📊 5-Fold CV Sonuclari:')
for i, s in enumerate(cv_scores, 1): print(f'  Fold {i}: {"█" * int(s*30)} %{s*100:.2f}')
print(f'\n🎯 CV Ort: %{cv_scores.mean()*100:.2f} | Std: ±%{cv_scores.std()*100:.2f}')
print(f'📌 Test Acc: %{acc*100:.2f}')

In [ ]:
# 7. GRIDSEARCHCV HIPERPARAMETRE OPTİMİZASYONU
param_grid = {
    'n_estimators'     : [100, 200],
    'max_depth'        : [3, 5, 7],
    'learning_rate'    : [0.05, 0.1, 0.2],
    'subsample'        : [0.7, 1.0],
    'colsample_bytree' : [0.7, 1.0],
}
grid = GridSearchCV(
    xgb.XGBClassifier(eval_metric='logloss', random_state=42, n_jobs=-1),
    param_grid,
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    scoring='accuracy', n_jobs=-1, verbose=1
)
print('🔍 GridSearchCV baslatiliyor...')
grid.fit(X_train, y_train_fit)
print(f'\n✅ En iyi parametreler:')
for k, v in grid.best_params_.items(): print(f'  {k:<20}: {v}')
best_xgb  = grid.best_estimator_
best_acc  = accuracy_score(y_test, best_xgb.predict(X_test))
print(f'\n🎯 CV best: %{grid.best_score_*100:.2f} | Test best: %{best_acc*100:.2f}')
print(f'📈 Iyilesme: %{(best_acc - acc)*100:.2f}')
print('\n✅ Kullanima hazir: best_xgb')